# 04C - SIMCA Candidate Concatenation and Memory-Safe Validation Refit

This notebook concatenates the model candidates produced by 04A and 04B, deduplicates them with the official SIMCA candidate key, optionally collapses equivalent refit configurations, and refits them on the fixed validation split.

The refit is batch-based. Global object/pixel/3-way metrics are kept as compact tables, while detailed projection tables are written incrementally by batch so the notebook does not keep all pixel projections in RAM.


## Inputs and outputs

Inputs:

- `results/04A_simca_grid_search_<RESULTS_TAG>/grid_model_candidates.parquet`
- `results/04B_simca_optuna_search_<RESULTS_TAG>/optuna_new_model_candidates.parquet`
- `results/03_pca_<RESULTS_TAG>/pca_selected_preprocessings.parquet`
- `HSI Data/processed/nir_uco_database.h5`

Main compact outputs:

- `candidate_panel.parquet`
- `refit_config_dedup_summary.parquet`
- `refit_config_duplicates.parquet`
- `metric_equivalent_config_groups.parquet`
- `metric_equivalent_config_dropped.parquet`
- `duplicated_refit_panel.parquet`
- `duplicated_refit_2way_object_metrics.parquet`
- `duplicated_refit_2way_pixel_metrics.parquet`
- `duplicated_refit_metric_comparison.parquet`
- `duplicated_refit_errors.parquet`
- `duplicated_refit_batch_manifest.parquet`
- `validation_refit_2way_object_metrics.parquet`
- `validation_refit_2way_pixel_metrics.parquet`
- `validation_3way_threshold_grid.parquet`
- `validation_3way_selected_thresholds.parquet`
- `validation_refit_3way_object_metrics.parquet`
- `validation_refit_metrics_long.parquet`
- `validation_refit_batch_manifest.parquet`
- `validation_refit_errors.parquet`
- `validation_refit_diagnostics.parquet`
- `validation_refit_protocol.parquet`

Detailed projection outputs are saved by batch under `validation_refit_batches/`. Combined object, pixel, and 3-way object projection tables are disabled by default to avoid RAM spikes; enable them only for a smaller candidate subset or when a downstream analysis explicitly needs a single merged table.


In [1]:
from pathlib import Path
import sys
import json
import gc

import numpy as np
import pandas as pd

CURRENT_DIR = Path.cwd().resolve()
if (CURRENT_DIR / "src").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "src").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise RuntimeError(
        "Could not find project root. Run this notebook from the project root or notebooks/."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

from src import experiment_config as expcfg
from src.io.database_h5 import load_nir_uco_h5
from src.utils import load_parquet, save_parquet, list_result_files
from src.workflows.simca_tables import compact_simca_table_for_path
from src.spectra.band_selection import (
    select_wavelength_range_from_database,
    wavelength_selection_summary,
)
from src.decision.metrics import summarize_pixel_errors_by_image
from src.decision.uncertainty import (
    add_three_way_confidence,
    calibrate_three_way_thresholds_by_config,
    evaluate_three_way_by_config,
)
from src.workflows.simca import (
    make_target_train_filters,
    refit_selected_simca_configs,
)
from src.workflows.simca_candidates import (
    add_selection_track,
    build_pca_preprocessing_configs_by_matrix_family,
    deduplicate_simca_candidates,
    deduplicate_metric_equivalent_simca_candidates,
    deduplicate_simca_refit_configs,
    filter_simca_candidates_by_pca_preprocessing,
    validate_simca_candidate_contract,
    validate_simca_evaluation_contract,
)
from src.workflows.simca_selection_utils import (
    add_detection_selection_score,
    fill_selected_config_defaults,
    materialize_selection_metrics,
    normalize_simca_rule_columns,
)


PROJECT_ROOT: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts


## Configuration

The refit uses the same validation protocol as 04A and 04B: train on pure target objects from batches 1-2, project pure reference objects from batch 3, and keep object/pixel outputs for downstream robustness analysis.

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DB_H5_PATH = PROJECT_ROOT / "HSI Data" / "processed" / "nir_uco_database.h5"

WAVELENGTH_MODE = expcfg.DEFAULT_WAVELENGTH_MODE
RESULTS_TAG = expcfg.DEFAULT_RESULTS_TAG
RESULTS_03_DIR = PROJECT_ROOT / "results" / f"03_pca_{RESULTS_TAG}"
RESULTS_04A_DIR = PROJECT_ROOT / "results" / f"04A_simca_grid_search_{RESULTS_TAG}"
RESULTS_04B_DIR = PROJECT_ROOT / "results" / f"04B_simca_optuna_search_{RESULTS_TAG}"
RESULTS_DIR = PROJECT_ROOT / "results" / f"04C_simca_concat_refit_{RESULTS_TAG}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PCA_SELECTED_PREPROCESSINGS_PATH = RESULTS_03_DIR / "pca_selected_preprocessings.parquet"
GRID_MODEL_CANDIDATES_PATH = RESULTS_04A_DIR / "grid_model_candidates.parquet"
OPTUNA_NEW_MODEL_CANDIDATES_PATH = RESULTS_04B_DIR / "optuna_new_model_candidates.parquet"

CANDIDATE_PANEL_PATH = RESULTS_DIR / "candidate_panel.parquet"
REFIT_CONFIG_DEDUP_SUMMARY_PATH = RESULTS_DIR / "refit_config_dedup_summary.parquet"
REFIT_CONFIG_DUPLICATES_PATH = RESULTS_DIR / "refit_config_duplicates.parquet"
METRIC_EQUIVALENCE_GROUPS_PATH = RESULTS_DIR / "metric_equivalent_config_groups.parquet"
METRIC_EQUIVALENCE_DROPPED_PATH = RESULTS_DIR / "metric_equivalent_config_dropped.parquet"
DUPLICATED_REFIT_PANEL_PATH = RESULTS_DIR / "duplicated_refit_panel.parquet"
DUPLICATED_REFIT_2WAY_OBJECT_METRICS_PATH = RESULTS_DIR / "duplicated_refit_2way_object_metrics.parquet"
DUPLICATED_REFIT_2WAY_PIXEL_METRICS_PATH = RESULTS_DIR / "duplicated_refit_2way_pixel_metrics.parquet"
DUPLICATED_REFIT_PIXEL_ERRORS_BY_IMAGE_PATH = RESULTS_DIR / "duplicated_refit_pixel_errors_by_image.parquet"
DUPLICATED_REFIT_METRIC_COMPARISON_PATH = RESULTS_DIR / "duplicated_refit_metric_comparison.parquet"
DUPLICATED_REFIT_ERRORS_PATH = RESULTS_DIR / "duplicated_refit_errors.parquet"
DUPLICATED_REFIT_BATCH_MANIFEST_PATH = RESULTS_DIR / "duplicated_refit_batch_manifest.parquet"
VALIDATION_2WAY_OBJECT_METRICS_PATH = RESULTS_DIR / "validation_refit_2way_object_metrics.parquet"
VALIDATION_2WAY_PIXEL_METRICS_PATH = RESULTS_DIR / "validation_refit_2way_pixel_metrics.parquet"
VALIDATION_3WAY_THRESHOLD_GRID_PATH = RESULTS_DIR / "validation_3way_threshold_grid.parquet"
VALIDATION_3WAY_SELECTED_THRESHOLDS_PATH = RESULTS_DIR / "validation_3way_selected_thresholds.parquet"
VALIDATION_3WAY_OBJECT_METRICS_PATH = RESULTS_DIR / "validation_refit_3way_object_metrics.parquet"
VALIDATION_METRICS_LONG_PATH = RESULTS_DIR / "validation_refit_metrics_long.parquet"
VALIDATION_OBJECTS_PATH = RESULTS_DIR / "validation_refit_objects.parquet"
VALIDATION_PIXELS_PATH = RESULTS_DIR / "validation_refit_pixels.parquet"
VALIDATION_3WAY_OBJECTS_PATH = RESULTS_DIR / "validation_refit_3way_objects.parquet"
VALIDATION_PIXEL_ERRORS_BY_IMAGE_PATH = RESULTS_DIR / "validation_refit_pixel_errors_by_image.parquet"
VALIDATION_REFIT_ERRORS_PATH = RESULTS_DIR / "validation_refit_errors.parquet"
VALIDATION_DIAGNOSTICS_PATH = RESULTS_DIR / "validation_refit_diagnostics.parquet"
VALIDATION_PROTOCOL_PATH = RESULTS_DIR / "validation_refit_protocol.parquet"
VALIDATION_BATCH_MANIFEST_PATH = RESULTS_DIR / "validation_refit_batch_manifest.parquet"
WAVELENGTH_CONFIG_PATH = RESULTS_DIR / "wavelength_config.parquet"
PREPROCESSING_SCOPE_PATH = RESULTS_DIR / "preprocessing_scope.parquet"

VALIDATION_BATCH_DIR = RESULTS_DIR / "validation_refit_batches"
VALIDATION_BATCH_METRICS_DIR = VALIDATION_BATCH_DIR / "metrics"
VALIDATION_BATCH_OBJECTS_DIR = VALIDATION_BATCH_DIR / "objects"
VALIDATION_BATCH_PIXELS_DIR = VALIDATION_BATCH_DIR / "pixels"
VALIDATION_BATCH_3WAY_OBJECTS_DIR = VALIDATION_BATCH_DIR / "objects_3way"
for _path in [
    VALIDATION_BATCH_DIR,
    VALIDATION_BATCH_METRICS_DIR,
    VALIDATION_BATCH_OBJECTS_DIR,
    VALIDATION_BATCH_PIXELS_DIR,
    VALIDATION_BATCH_3WAY_OBJECTS_DIR,
]:
    _path.mkdir(parents=True, exist_ok=True)

TARGET_CLASS = expcfg.TARGET_CLASS
NON_TARGET_LABEL = expcfg.NON_TARGET_LABEL
REFERENCE_CLASSES = list(expcfg.REFERENCE_CLASSES)

TRAIN_FILTERS = make_target_train_filters(
    target_class=TARGET_CLASS,
    train_batches=expcfg.SIMCA_TRAIN_BATCHES,
)
VALIDATION_FILTERS = {
    "sample_kind": ["pure"],
    "object_nut_type": REFERENCE_CLASSES,
    "batch": list(expcfg.SIMCA_VALIDATION_BATCHES),
}

USE_WAVELENGTH_WINDOW = False
WINDOW_MIN_NM = 1225.0
WINDOW_MAX_NM = 1675.0

RUN_REFIT = True
USE_EXISTING_VALIDATION_REFIT_OUTPUTS = True
RANDOM_STATE = expcfg.RANDOM_STATE
REPLACE_BALANCED_PIXELS = expcfg.REPLACE_BALANCED_PIXELS
CV_N_SPLITS = expcfg.CV_N_SPLITS
CV_GROUP_COL = expcfg.CV_GROUP_COL

# Memory controls.
REFIT_BATCH_SIZE = 50
MAX_REFIT_CANDIDATES = None
SAVE_BATCH_METRIC_TABLES = True
SAVE_BATCH_OBJECT_TABLES = True
SAVE_BATCH_PIXEL_TABLES = True
SAVE_BATCH_3WAY_OBJECT_TABLES = True
DEDUPLICATE_REFIT_CONFIGS = True
REFIT_CONFIG_DEDUP_COLUMNS = list(expcfg.SIMCA_REFIT_CONFIG_DEDUP_COLUMNS)
DROP_METRIC_EQUIVALENT_CONFIGS = True
REFIT_DUPLICATED = False
DUPLICATED_REFIT_BATCH_SIZE = 50
DUPLICATED_REFIT_MAX_GROUPS = None
METRIC_EQUIVALENCE_METRIC_COLUMNS = list(expcfg.SIMCA_METRIC_EQUIVALENCE_METRIC_COLUMNS)
METRIC_EQUIVALENCE_PROTECTED_COLUMNS = list(expcfg.SIMCA_METRIC_EQUIVALENCE_PROTECTED_COLUMNS)
METRIC_EQUIVALENCE_PARAMETER_GROUPS = dict(expcfg.SIMCA_METRIC_EQUIVALENCE_PARAMETER_GROUPS)
METRIC_EQUIVALENCE_PREFERENCE_NOTE = (
    "Rows are sorted before collapse; the first row is kept. "
    "Tie-breaks prefer lower FN/FP, higher balanced accuracy, multi-source candidates, "
    "lower n_components, lower preprocessing complexity, and lower object_threshold."
)
SAVE_COMBINED_OBJECT_TABLES = True
SAVE_COMBINED_PIXEL_TABLES = True
SAVE_COMBINED_3WAY_OBJECT_TABLES = True

THREE_WAY_LOWER_THRESHOLDS = np.round(np.arange(0.05, 0.61, 0.05), 2)
THREE_WAY_UPPER_THRESHOLDS = np.round(np.arange(0.40, 0.96, 0.05), 2)
MAX_THREE_WAY_TARGET_MISS_RATE = 0.00
MAX_THREE_WAY_FALSE_ACCEPT_RATE = 0.30
MAX_THREE_WAY_UNCERTAIN_RATE = 0.60

track_specs_df = pd.DataFrame([
    {"selection_track": track, **spec}
    for track, spec in expcfg.SIMCA_SELECTION_TRACK_SPECS.items()
])

display(track_specs_df)
print("DB_H5_PATH:", DB_H5_PATH)
print("GRID_MODEL_CANDIDATES_PATH:", GRID_MODEL_CANDIDATES_PATH)
print("OPTUNA_NEW_MODEL_CANDIDATES_PATH:", OPTUNA_NEW_MODEL_CANDIDATES_PATH)
print("RESULTS_DIR:", RESULTS_DIR)
print("RUN_REFIT:", RUN_REFIT)
print("USE_EXISTING_VALIDATION_REFIT_OUTPUTS:", USE_EXISTING_VALIDATION_REFIT_OUTPUTS)
print("REFIT_BATCH_SIZE:", REFIT_BATCH_SIZE)
print("SAVE_BATCH_PIXEL_TABLES:", SAVE_BATCH_PIXEL_TABLES)
print("DEDUPLICATE_REFIT_CONFIGS:", DEDUPLICATE_REFIT_CONFIGS)
print("REFIT_CONFIG_DEDUP_COLUMNS:", REFIT_CONFIG_DEDUP_COLUMNS)
print("DROP_METRIC_EQUIVALENT_CONFIGS:", DROP_METRIC_EQUIVALENT_CONFIGS)
print("REFIT_DUPLICATED:", REFIT_DUPLICATED)
print("DUPLICATED_REFIT_BATCH_SIZE:", DUPLICATED_REFIT_BATCH_SIZE)
print("DUPLICATED_REFIT_MAX_GROUPS:", DUPLICATED_REFIT_MAX_GROUPS)
print("METRIC_EQUIVALENCE_PARAMETER_GROUPS:", METRIC_EQUIVALENCE_PARAMETER_GROUPS)
print("SAVE_COMBINED_OBJECT_TABLES:", SAVE_COMBINED_OBJECT_TABLES)
print("SAVE_COMBINED_PIXEL_TABLES:", SAVE_COMBINED_PIXEL_TABLES)
print("SAVE_COMBINED_3WAY_OBJECT_TABLES:", SAVE_COMBINED_3WAY_OBJECT_TABLES)


## Load inputs

In [3]:
required_paths = [
    DB_H5_PATH,
    PCA_SELECTED_PREPROCESSINGS_PATH,
    GRID_MODEL_CANDIDATES_PATH,
    OPTUNA_NEW_MODEL_CANDIDATES_PATH,
]
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError("Missing required input file(s): " + ", ".join(map(str, missing_paths)))

object_db, image_db = load_nir_uco_h5(
    DB_H5_PATH,
    reconstruct_heavy_object_arrays=True,
)

if USE_WAVELENGTH_WINDOW:
    object_db, image_db, wavelengths, wavelength_info = select_wavelength_range_from_database(
        object_db=object_db,
        image_db=image_db,
        min_nm=WINDOW_MIN_NM,
        max_nm=WINDOW_MAX_NM,
    )
    wavelength_selection_df = wavelength_selection_summary(wavelength_info)
else:
    first_obj = next(iter(object_db.values()))
    wavelengths = first_obj.get("wavelengths")
    wavelengths = np.asarray(wavelengths, dtype=float) if wavelengths is not None else None
    wavelength_selection_df = pd.DataFrame()

if wavelengths is None:
    raise RuntimeError("No wavelength axis found in object_db.")

pca_selected_preprocessings_df = load_parquet(PCA_SELECTED_PREPROCESSINGS_PATH)
preprocessing_configs_by_family = build_pca_preprocessing_configs_by_matrix_family(
    pca_selected_preprocessings_df
)

preprocessing_scope_df = pd.DataFrame([
    {
        "matrix_family": family,
        "preprocessing": name,
        "preprocessing_steps": "+".join(steps),
    }
    for family, configs in preprocessing_configs_by_family.items()
    for name, steps in configs.items()
])

grid_model_candidates_df = load_parquet(GRID_MODEL_CANDIDATES_PATH)
optuna_new_model_candidates_df = load_parquet(OPTUNA_NEW_MODEL_CANDIDATES_PATH)

wavelength_config_df = pd.DataFrame([{
    "wavelength_mode": WAVELENGTH_MODE,
    "use_wavelength_window": bool(USE_WAVELENGTH_WINDOW),
    "results_tag": RESULTS_TAG,
    "window_min_nm": WINDOW_MIN_NM if USE_WAVELENGTH_WINDOW else np.nan,
    "window_max_nm": WINDOW_MAX_NM if USE_WAVELENGTH_WINDOW else np.nan,
    "n_bands": int(len(wavelengths)),
    "min_wavelength_nm": float(np.min(wavelengths)),
    "max_wavelength_nm": float(np.max(wavelengths)),
}])

save_parquet(compact_simca_table_for_path(wavelength_config_df, WAVELENGTH_CONFIG_PATH), WAVELENGTH_CONFIG_PATH)
save_parquet(compact_simca_table_for_path(preprocessing_scope_df, PREPROCESSING_SCOPE_PATH), PREPROCESSING_SCOPE_PATH)

print("Number of images:", len(image_db))
print("Number of objects:", len(object_db))
print("04A grid candidates:", grid_model_candidates_df.shape)
print("04B new Optuna candidates:", optuna_new_model_candidates_df.shape)
display(wavelength_config_df)
display(preprocessing_scope_df.sort_values(["matrix_family", "preprocessing"]))

Number of images: 48
Number of objects: 1262
04A grid candidates: (3240, 44)
04B new Optuna candidates: (966, 44)


,wavelength_mode,use_wavelength_window,results_tag,window_min_nm,window_max_nm,n_bands,min_wavelength_nm,max_wavelength_nm
0,non_noisy_all,False,non_noisy_all,NaN,NaN,63,960.735294,1702.0


,matrix_family,preprocessing,preprocessing_steps
0,object_matrix,absorbance_sg_d1,absorbance+sg_d1
1,object_matrix,absorbance_sg_d2,absorbance+sg_d2
4,object_matrix,absorbance_snv_sg_d1,absorbance+snv+sg_d1
2,object_matrix,absorbance_snv_sg_d2,absorbance+snv+sg_d2
3,object_matrix,snv_sg_d2,snv+sg_d2
6,pixel_matrix,absorbance_snv_sg_smooth,absorbance+snv+sg_smooth
9,pixel_matrix,raw,raw
8,pixel_matrix,sg_smooth,sg_smooth
7,pixel_matrix,snv,snv
5,pixel_matrix,snv_sg_smooth,snv+sg_smooth


## Build candidate panel

The candidate panel is the single source passed to the validation refit. It keeps the stable `candidate_id`, assigns a 04C-local `selected_config_id`, preserves source provenance, collapses exact refit duplicates, and can remove metric-equivalent candidates before the costly projection step.


In [4]:
candidate_parts = []

if len(grid_model_candidates_df) > 0:
    grid_part = grid_model_candidates_df.copy()
    grid_part["candidate_source"] = "04A_grid_search"
    candidate_parts.append(grid_part)

if len(optuna_new_model_candidates_df) > 0:
    optuna_part = optuna_new_model_candidates_df.copy()
    optuna_part["candidate_source"] = "04B_optuna_search"
    candidate_parts.append(optuna_part)

if not candidate_parts:
    raise RuntimeError("No model candidates found in 04A/04B inputs.")

candidate_panel_df = pd.concat(candidate_parts, ignore_index=True, sort=False)
if "selected_config_id" in candidate_panel_df.columns:
    candidate_panel_df["source_selected_config_id"] = candidate_panel_df["selected_config_id"].astype(str)

candidate_panel_df = normalize_simca_rule_columns(candidate_panel_df)
candidate_panel_df = fill_selected_config_defaults(
    candidate_panel_df,
    default_values={
        "target_class": TARGET_CLASS,
        "non_target_label": NON_TARGET_LABEL,
        "sg_window_length": 11,
        "sg_polyorder": 2,
        "position_dilation_radius": 3,
        "m": expcfg.M_BALANCED_PIXELS,
        "alpha": expcfg.SIMCA_ALPHA_VALUES[0],
        "object_threshold": expcfg.SIMCA_OBJECT_THRESHOLDS[0],
    },
)
candidate_panel_df = filter_simca_candidates_by_pca_preprocessing(
    candidate_panel_df,
    pca_selected_preprocessings_df,
    strict=True,
)
candidate_panel_df = deduplicate_simca_candidates(candidate_panel_df)
candidate_panel_df["model_candidate_id"] = candidate_panel_df["candidate_id"]

candidate_panel_df["_preprocessing_n_steps"] = (
    candidate_panel_df["preprocessing_steps"]
    .astype(str)
    .str.split("+")
    .map(len)
    if "preprocessing_steps" in candidate_panel_df.columns
    else 999
)
if "n_candidate_sources" in candidate_panel_df.columns:
    candidate_panel_df["_source_priority"] = pd.to_numeric(
        candidate_panel_df["n_candidate_sources"],
        errors="coerce",
    ).fillna(1)
else:
    candidate_panel_df["_source_priority"] = (
        candidate_panel_df.get("candidate_sources", "unknown")
        .astype(str)
        .str.split(",")
        .map(len)
    )

sort_cols = [
    col for col in [
        "matrix_family",
        "fn_rate",
        "fp_rate",
        "balanced_accuracy",
        "_source_priority",
        "n_components",
        "_preprocessing_n_steps",
        "object_threshold",
        "preprocessing",
        "rule_for_refit",
        "candidate_id",
    ]
    if col in candidate_panel_df.columns
]
ascending_by_col = {
    "balanced_accuracy": False,
    "_source_priority": False,
}
ascending = [
    ascending_by_col.get(col, True)
    for col in sort_cols
]
candidate_panel_df = (
    candidate_panel_df
    .sort_values(sort_cols, ascending=ascending)
    .reset_index(drop=True)
)

candidate_panel_before_refit_dedup_df = candidate_panel_df.copy()
if DEDUPLICATE_REFIT_CONFIGS:
    (
        candidate_panel_df,
        refit_config_duplicates_df,
        refit_config_dedup_summary_df,
    ) = deduplicate_simca_refit_configs(
        candidate_panel_df,
        key_columns=REFIT_CONFIG_DEDUP_COLUMNS,
        strict=True,
    )
else:
    refit_config_duplicates_df = pd.DataFrame(columns=candidate_panel_df.columns)
    refit_config_dedup_summary_df = pd.DataFrame()

candidate_panel_after_refit_dedup_df = candidate_panel_df.copy()
if DROP_METRIC_EQUIVALENT_CONFIGS:
    (
        candidate_panel_df,
        metric_equivalence_dropped_df,
        metric_equivalence_groups_df,
    ) = deduplicate_metric_equivalent_simca_candidates(
        candidate_panel_df,
        metric_columns=METRIC_EQUIVALENCE_METRIC_COLUMNS,
        parameter_groups=METRIC_EQUIVALENCE_PARAMETER_GROUPS,
        protected_columns=METRIC_EQUIVALENCE_PROTECTED_COLUMNS,
        strict=True,
    )
else:
    metric_equivalence_dropped_df = pd.DataFrame(columns=candidate_panel_df.columns)
    metric_equivalence_groups_df = pd.DataFrame()

candidate_panel_df = candidate_panel_df.reset_index(drop=True)
candidate_panel_df["selected_config_id"] = [
    f"04C_refit_{i:06d}" for i in range(len(candidate_panel_df))
]

duplicated_kept_df = candidate_panel_df.loc[
    candidate_panel_df.get("metric_equivalence_group_id", pd.Series(index=candidate_panel_df.index, dtype="object")).notna()
].copy()
duplicated_kept_df["duplicated_refit_role"] = "kept"
duplicated_dropped_df = metric_equivalence_dropped_df.copy()
if len(duplicated_dropped_df) > 0:
    duplicated_dropped_df["duplicated_refit_role"] = "dropped"

refit_panel_duplicated_df = pd.concat(
    [duplicated_kept_df, duplicated_dropped_df],
    ignore_index=True,
    sort=False,
)
if len(refit_panel_duplicated_df) > 0:
    for col in METRIC_EQUIVALENCE_METRIC_COLUMNS:
        if col in refit_panel_duplicated_df.columns:
            refit_panel_duplicated_df[f"pre_refit_{col}"] = refit_panel_duplicated_df[col]
    if "selected_config_id" in refit_panel_duplicated_df.columns:
        refit_panel_duplicated_df["main_selected_config_id"] = (
            refit_panel_duplicated_df["selected_config_id"].astype("string")
        )
    duplicated_sort_cols = [
        col for col in [
            "metric_equivalence_group_id",
            "duplicated_refit_role",
            "metric_equivalence_original_order",
            "candidate_id",
        ]
        if col in refit_panel_duplicated_df.columns
    ]
    if duplicated_sort_cols:
        refit_panel_duplicated_df = (
            refit_panel_duplicated_df
            .sort_values(duplicated_sort_cols)
            .reset_index(drop=True)
        )
    refit_panel_duplicated_df["selected_config_id"] = [
        f"04C_dup_refit_{i:06d}" for i in range(len(refit_panel_duplicated_df))
    ]
else:
    refit_panel_duplicated_df = pd.DataFrame(columns=candidate_panel_df.columns)

candidate_panel_df = candidate_panel_df.drop(
    columns=[
        "_preprocessing_n_steps",
        "_source_priority",
    ],
    errors="ignore",
)
metric_equivalence_dropped_df = metric_equivalence_dropped_df.drop(
    columns=[
        "_preprocessing_n_steps",
        "_source_priority",
    ],
    errors="ignore",
)
refit_panel_duplicated_df = refit_panel_duplicated_df.drop(
    columns=[
        "_preprocessing_n_steps",
        "_source_priority",
    ],
    errors="ignore",
)

validate_simca_candidate_contract(candidate_panel_df)
if len(refit_panel_duplicated_df) > 0:
    validate_simca_candidate_contract(refit_panel_duplicated_df)
save_parquet(compact_simca_table_for_path(candidate_panel_df, CANDIDATE_PANEL_PATH), CANDIDATE_PANEL_PATH)
save_parquet(compact_simca_table_for_path(refit_panel_duplicated_df, DUPLICATED_REFIT_PANEL_PATH), DUPLICATED_REFIT_PANEL_PATH)
save_parquet(compact_simca_table_for_path(refit_config_dedup_summary_df, REFIT_CONFIG_DEDUP_SUMMARY_PATH), REFIT_CONFIG_DEDUP_SUMMARY_PATH)
save_parquet(compact_simca_table_for_path(refit_config_duplicates_df, REFIT_CONFIG_DUPLICATES_PATH), REFIT_CONFIG_DUPLICATES_PATH)
save_parquet(compact_simca_table_for_path(metric_equivalence_groups_df, METRIC_EQUIVALENCE_GROUPS_PATH), METRIC_EQUIVALENCE_GROUPS_PATH)
save_parquet(compact_simca_table_for_path(metric_equivalence_dropped_df, METRIC_EQUIVALENCE_DROPPED_PATH), METRIC_EQUIVALENCE_DROPPED_PATH)

print("Candidate panel before refit-config dedup:", candidate_panel_before_refit_dedup_df.shape)
print("Candidate panel after refit-config dedup:", candidate_panel_after_refit_dedup_df.shape)
print("Candidate panel after metric-equivalence pruning:", candidate_panel_df.shape)
print("Refit-config duplicate rows removed:", len(refit_config_duplicates_df))
print("Metric-equivalent rows removed:", len(metric_equivalence_dropped_df))
print("Duplicated refit check panel:", refit_panel_duplicated_df.shape)
display(candidate_panel_df["matrix_family"].value_counts(dropna=False))
display(candidate_panel_df["candidate_sources"].value_counts(dropna=False))
if len(refit_config_dedup_summary_df) > 0:
    display(
        refit_config_dedup_summary_df["n_refit_config_candidates"]
        .value_counts()
        .sort_index()
        .rename_axis("n_candidates_per_refit_config")
        .reset_index(name="n_refit_configs")
    )
if len(metric_equivalence_groups_df) > 0:
    display(
        metric_equivalence_groups_df["varied_parameter_group"]
        .value_counts()
        .rename_axis("varied_parameter_group")
        .reset_index(name="n_groups")
    )
display(candidate_panel_df.head())
if len(refit_config_duplicates_df) > 0:
    display(refit_config_duplicates_df.head())
if len(metric_equivalence_dropped_df) > 0:
    display(metric_equivalence_dropped_df.head())
if len(refit_panel_duplicated_df) > 0:
    display(refit_panel_duplicated_df.head())


Candidate panel before refit-config dedup: (4206, 64)
Candidate panel after refit-config dedup: (4149, 69)
Candidate panel after metric-equivalence pruning: (1982, 72)
Refit-config duplicate rows removed: 57
Metric-equivalent rows removed: 2167
Duplicated refit check panel: (2412, 82)


matrix_family
pixel_matrix     1099
object_matrix     883
Name: count, dtype: int64

candidate_sources
04A_grid_search                      1076
04B_optuna_search                     858
04A_grid_search,04B_optuna_search      48
Name: count, dtype: int64

,n_candidates_per_refit_config,n_refit_configs
0,1,4092
1,2,57


,varied_parameter_group,n_groups
0,object_threshold,954
1,preprocessing,244
2,matrix_method,132
3,rule,125
4,n_components,102
5,position_dilation,31
6,savgol,4


,selected_config_id,candidate_id,model_candidate_id,matrix_family,target_class,non_target_label,model_family,matrix_method,training_matrix_id,m_effective,...,refit_config_id,refit_config_duplicate_rank,n_refit_config_candidates,refit_config_candidate_ids,refit_config_candidate_sources,metric_equivalence_original_order,metric_equivalence_group_id,metric_equivalence_kept_candidate_id,metric_equivalence_varied_parameter_group,metric_equivalence_drop_reason
0,04C_refit_000000,simca_0d9ef35630459e28,simca_0d9ef35630459e28,object_matrix,peanut,almond,rule_variant_grid,object_median,object_median,40.0,...,refitcfg_05835c8f28287c66,1,1,simca_0d9ef35630459e28,04B_optuna_search,0,metric_eq_001331,NaN,NaN,NaN
1,04C_refit_000001,simca_f21f2ff62531f442,simca_f21f2ff62531f442,object_matrix,peanut,almond,rule_variant_grid,object_median,object_median,40.0,...,refitcfg_11ccb5c67c30a9d9,1,1,simca_f21f2ff62531f442,04B_optuna_search,2,NaN,NaN,NaN,NaN
2,04C_refit_000002,simca_a207c12cce354da3,simca_a207c12cce354da3,object_matrix,peanut,almond,rule_variant_grid,object_median,object_median,40.0,...,refitcfg_941887f821ed3beb,1,1,simca_a207c12cce354da3,04B_optuna_search,3,NaN,NaN,NaN,NaN
3,04C_refit_000003,simca_08a074054db71001,simca_08a074054db71001,object_matrix,peanut,almond,rule_variant_grid,object_median,object_median,40.0,...,refitcfg_660f259518000307,1,1,simca_08a074054db71001,04B_optuna_search,4,NaN,NaN,NaN,NaN
4,04C_refit_000004,simca_138c3b542a51f39b,simca_138c3b542a51f39b,object_matrix,peanut,almond,rule_variant_grid,object_median,object_median,40.0,...,refitcfg_51a2f6db9d7da42e,1,1,simca_138c3b542a51f39b,04B_optuna_search,5,metric_eq_001562,NaN,NaN,NaN


,selected_config_id,candidate_id,model_candidate_id,matrix_family,target_class,non_target_label,model_family,matrix_method,training_matrix_id,m_effective,...,candidate_sources,n_candidate_sources,n_duplicate_rows,_preprocessing_n_steps,_source_priority,refit_config_id,refit_config_duplicate_rank,n_refit_config_candidates,refit_config_candidate_ids,refit_config_candidate_sources
0,optuna_0234,simca_53fda171b13b6751,simca_53fda171b13b6751,object_matrix,peanut,almond,rule_variant_grid,object_median,object_median,40.0,...,"04A_grid_search,04B_optuna_search",1,1,2,1,refitcfg_985add5c5b16fa12,2,2,"simca_1c2855ffea473c9e,simca_53fda171b13b6751","04A_grid_search,04B_optuna_search"
1,optuna_0101,simca_500371023a391865,simca_500371023a391865,object_matrix,peanut,almond,rule_variant_grid,object_median,object_median,40.0,...,"04A_grid_search,04B_optuna_search",1,1,2,1,refitcfg_88d7bfb7efac8f71,2,2,"simca_262a43cd828bd2b3,simca_500371023a391865","04A_grid_search,04B_optuna_search"
2,optuna_0190,simca_d8914731e44465d9,simca_d8914731e44465d9,object_matrix,peanut,almond,rule_variant_grid,object_median,object_median,40.0,...,"04A_grid_search,04B_optuna_search",1,1,2,1,refitcfg_b47ea973236ab2ae,2,2,"simca_331f03da21b0b2d2,simca_d8914731e44465d9","04A_grid_search,04B_optuna_search"
3,04A_grid_000840,simca_fe19269a5c02b86b,simca_fe19269a5c02b86b,object_matrix,peanut,almond,empirical_cv_rule,object_median,object_median,40.0,...,"04A_grid_search,04B_optuna_search",1,1,2,1,refitcfg_f663e3dfa3294f51,2,2,"simca_59f9db3ccd08b3aa,simca_fe19269a5c02b86b","04A_grid_search,04B_optuna_search"
4,optuna_0208,simca_ee3fd0b54f5113c3,simca_ee3fd0b54f5113c3,object_matrix,peanut,almond,rule_variant_grid,object_median,object_median,40.0,...,"04A_grid_search,04B_optuna_search",1,1,2,1,refitcfg_69f7752af77ff667,2,2,"simca_9202e16661d5e121,simca_ee3fd0b54f5113c3","04A_grid_search,04B_optuna_search"


,selected_config_id,candidate_id,model_candidate_id,matrix_family,target_class,non_target_label,model_family,matrix_method,training_matrix_id,m_effective,...,refit_config_id,refit_config_duplicate_rank,n_refit_config_candidates,refit_config_candidate_ids,refit_config_candidate_sources,metric_equivalence_original_order,metric_equivalence_group_id,metric_equivalence_kept_candidate_id,metric_equivalence_varied_parameter_group,metric_equivalence_drop_reason
0,optuna_0273,simca_774ca06be5b4fd2a,simca_774ca06be5b4fd2a,object_matrix,peanut,almond,rule_variant_grid,object_median,object_median,40.0,...,refitcfg_530f90ecfc9c34be,1,1,simca_774ca06be5b4fd2a,04B_optuna_search,1,metric_eq_001331,simca_0d9ef35630459e28,rule,Identical metrics and only 'rule' differs.
1,optuna_0124,simca_2d007dbe64ad8e7d,simca_2d007dbe64ad8e7d,object_matrix,peanut,almond,rule_variant_grid,object_median,object_median,40.0,...,refitcfg_c9f15a42754334e4,1,1,simca_2d007dbe64ad8e7d,04B_optuna_search,6,metric_eq_001562,simca_138c3b542a51f39b,position_dilation,Identical metrics and only 'position_dilation'...
2,optuna_0045,simca_bbadeacf96728c05,simca_bbadeacf96728c05,object_matrix,peanut,almond,rule_variant_grid,object_median,object_median,40.0,...,refitcfg_8868ae86f5faf5fc,1,1,simca_bbadeacf96728c05,04B_optuna_search,13,metric_eq_001563,simca_b2b3d8faa0b441bd,position_dilation,Identical metrics and only 'position_dilation'...
3,04A_grid_000814,simca_b25bb4bccf968f44,simca_b25bb4bccf968f44,object_matrix,peanut,almond,empirical_cv_rule,object_median,object_median,40.0,...,refitcfg_7f0bae3fa5b3353c,1,1,simca_b25bb4bccf968f44,04A_grid_search,16,metric_eq_001332,simca_15839f0dc189fcc4,rule,Identical metrics and only 'rule' differs.
4,04A_grid_000816,simca_10584c1e58085ac0,simca_10584c1e58085ac0,object_matrix,peanut,almond,empirical_cv_rule,object_median,object_median,40.0,...,refitcfg_9502f47de7cef5f9,1,1,simca_10584c1e58085ac0,04A_grid_search,19,metric_eq_001333,simca_a8d5b556d0a97e0b,rule,Identical metrics and only 'rule' differs.


,selected_config_id,candidate_id,model_candidate_id,matrix_family,target_class,non_target_label,model_family,matrix_method,training_matrix_id,m_effective,...,duplicated_refit_role,pre_refit_n,pre_refit_tp,pre_refit_fn,pre_refit_fp,pre_refit_tn,pre_refit_fn_rate,pre_refit_fp_rate,pre_refit_balanced_accuracy,main_selected_config_id
0,04C_dup_refit_000000,simca_3e0a6163cae88957,simca_3e0a6163cae88957,object_matrix,peanut,almond,empirical_cv_rule,object_mean,object_mean,40.0,...,dropped,108.0,1.0,52.0,0.0,55.0,0.981132,0.000000,0.509434,04A_grid_000095
1,04C_dup_refit_000001,simca_2069b62a8c25ad00,simca_2069b62a8c25ad00,object_matrix,peanut,almond,empirical_cv_rule,object_mean,object_mean,40.0,...,kept,108.0,1.0,52.0,0.0,55.0,0.981132,0.000000,0.509434,04C_refit_000695
2,04C_dup_refit_000002,simca_06dcb8926f51cc88,simca_06dcb8926f51cc88,object_matrix,peanut,almond,empirical_cv_rule,object_median,object_median,40.0,...,dropped,108.0,1.0,52.0,1.0,54.0,0.981132,0.018182,0.500343,04A_grid_001317
3,04C_dup_refit_000003,simca_861053d36ae8ee7d,simca_861053d36ae8ee7d,object_matrix,peanut,almond,empirical_cv_rule,object_mean,object_mean,40.0,...,dropped,108.0,0.0,53.0,0.0,55.0,1.000000,0.000000,0.500000,04A_grid_000119
4,04C_dup_refit_000004,simca_c538cb99323dff48,simca_c538cb99323dff48,object_matrix,peanut,almond,empirical_cv_rule,object_mean,object_mean,40.0,...,dropped,108.0,0.0,53.0,0.0,55.0,1.000000,0.000000,0.500000,04A_grid_000137


## Batch validation refit

Each batch is refit, scored at object and pixel level, calibrated for 3-way decisions, saved, and then released from memory. The consolidated tables kept in memory are metric-level tables; detailed object and 3-way object projections are available as batch parts, and detailed pixel projections stay disabled by default.

To reuse a previous validation refit and run only downstream optional checks, set `RUN_REFIT = False` and `USE_EXISTING_VALIDATION_REFIT_OUTPUTS = True` in the configuration cell.


In [ ]:
EVALUATION_STAGE = "validation_batch_3_refit"


def iter_candidate_batches(df, batch_size):
    if batch_size is None or int(batch_size) <= 0:
        raise ValueError("REFIT_BATCH_SIZE must be a positive integer.")
    batch_size = int(batch_size)
    for batch_idx, start in enumerate(range(0, len(df), batch_size), start=1):
        stop = min(start + batch_size, len(df))
        yield f"batch_{batch_idx:04d}", start, stop, df.iloc[start:stop].copy()


def concat_nonempty(parts):
    parts = [part for part in parts if part is not None and len(part) > 0]
    if not parts:
        return pd.DataFrame()
    return pd.concat(parts, ignore_index=True, sort=False)


def save_batch_table(df, directory, batch_id, stem):
    if df is None or len(df) == 0:
        return None
    path = directory / f"{batch_id}_{stem}.parquet"
    return save_parquet(compact_simca_table_for_path(df, path), path)



def read_simca_parquet_if_exists(path):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    return compact_simca_table_for_path(pd.read_parquet(path), path)


def required_existing_validation_refit_paths():
    return [
        VALIDATION_2WAY_OBJECT_METRICS_PATH,
        VALIDATION_2WAY_PIXEL_METRICS_PATH,
        VALIDATION_3WAY_OBJECT_METRICS_PATH,
    ]


def missing_existing_validation_refit_paths():
    return [
        path for path in required_existing_validation_refit_paths()
        if not Path(path).exists()
    ]


def load_existing_validation_refit_outputs():
    validation_2way_object_metrics_df = read_simca_parquet_if_exists(VALIDATION_2WAY_OBJECT_METRICS_PATH)
    validation_2way_pixel_metrics_df = read_simca_parquet_if_exists(VALIDATION_2WAY_PIXEL_METRICS_PATH)
    validation_3way_threshold_grid_df = read_simca_parquet_if_exists(VALIDATION_3WAY_THRESHOLD_GRID_PATH)
    validation_3way_selected_thresholds_df = read_simca_parquet_if_exists(VALIDATION_3WAY_SELECTED_THRESHOLDS_PATH)
    validation_3way_object_metrics_df = read_simca_parquet_if_exists(VALIDATION_3WAY_OBJECT_METRICS_PATH)

    validation_metrics_long_df = read_simca_parquet_if_exists(VALIDATION_METRICS_LONG_PATH)
    if validation_metrics_long_df.empty:
        validation_metrics_long_df = pd.concat(
            [
                validation_2way_object_metrics_df,
                validation_2way_pixel_metrics_df,
                validation_3way_object_metrics_df,
            ],
            ignore_index=True,
            sort=False,
        )

    return {
        "validation_refit_metrics_df": validation_metrics_long_df.copy(),
        "validation_2way_object_metrics_df": validation_2way_object_metrics_df,
        "validation_2way_pixel_metrics_df": validation_2way_pixel_metrics_df,
        "validation_3way_threshold_grid_df": validation_3way_threshold_grid_df,
        "validation_3way_selected_thresholds_df": validation_3way_selected_thresholds_df,
        "validation_3way_object_metrics_df": validation_3way_object_metrics_df,
        "validation_refit_pixel_errors_by_image_df": read_simca_parquet_if_exists(VALIDATION_PIXEL_ERRORS_BY_IMAGE_PATH),
        "validation_refit_errors_df": read_simca_parquet_if_exists(VALIDATION_REFIT_ERRORS_PATH),
        "validation_refit_objects_df": read_simca_parquet_if_exists(VALIDATION_OBJECTS_PATH),
        "validation_refit_pixels_df": read_simca_parquet_if_exists(VALIDATION_PIXELS_PATH),
        "validation_refit_3way_objects_df": read_simca_parquet_if_exists(VALIDATION_3WAY_OBJECTS_PATH),
        "validation_batch_manifest_df": read_simca_parquet_if_exists(VALIDATION_BATCH_MANIFEST_PATH),
    }


def finalize_metric_table(df, decision_mode, metric_level, add_score=True):
    if df is None or len(df) == 0:
        return pd.DataFrame()
    out = df.copy()
    out["decision_mode"] = decision_mode
    out["evaluation_stage"] = EVALUATION_STAGE
    out["metric_level"] = metric_level
    out = add_selection_track(out)
    if add_score:
        out = add_detection_selection_score(out)
    validate_simca_evaluation_contract(out)
    return out


def candidate_metadata_for_thresholds(panel_df):
    cols = [
        "selected_config_id",
        "candidate_id",
        "model_candidate_id",
        "matrix_family",
        "matrix_method",
        "training_matrix_id",
        "preprocessing",
        "rule_variant",
        "n_components",
        "alpha",
        "object_threshold",
        "candidate_sources",
    ]
    cols = [col for col in cols if col in panel_df.columns]
    return panel_df[cols].drop_duplicates("selected_config_id")


def build_2way_object_metrics(refit_metrics_df):
    metrics_df = materialize_selection_metrics(refit_metrics_df)
    return finalize_metric_table(metrics_df, decision_mode="2way", metric_level="object")


def build_2way_pixel_metrics(pixel_df):
    if pixel_df is None or len(pixel_df) == 0:
        return pd.DataFrame()

    pixel_group_cols = [
        "selected_config_id",
        "candidate_id",
        "model_candidate_id",
        "matrix_family",
        "matrix_method",
        "training_matrix_id",
        "preprocessing",
        "rule_variant",
        "n_components",
        "alpha",
        "object_threshold",
    ]
    pixel_group_cols = [
        col for col in pixel_group_cols
        if col in pixel_df.columns
    ]
    if not pixel_group_cols:
        raise RuntimeError("Pixel metrics require at least one grouping column.")

    metrics_df = summarize_pixel_errors_by_image(
        pixel_df,
        target_class=TARGET_CLASS,
        non_target_label=NON_TARGET_LABEL,
        group_cols=pixel_group_cols,
        sort_worst_first=False,
    )
    return finalize_metric_table(metrics_df, decision_mode="2way", metric_level="pixel")

def build_3way_outputs(object_df, panel_df):
    if object_df is None or len(object_df) == 0:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    threshold_grid_df, selected_thresholds_df = calibrate_three_way_thresholds_by_config(
        object_df=object_df,
        config_cols=["selected_config_id"],
        target_class=TARGET_CLASS,
        non_target_label=NON_TARGET_LABEL,
        lower_thresholds=THREE_WAY_LOWER_THRESHOLDS,
        upper_thresholds=THREE_WAY_UPPER_THRESHOLDS,
        max_target_miss_rate=MAX_THREE_WAY_TARGET_MISS_RATE,
        max_false_accept_rate=MAX_THREE_WAY_FALSE_ACCEPT_RATE,
        max_uncertain_rate=MAX_THREE_WAY_UNCERTAIN_RATE,
    )

    if selected_thresholds_df.empty:
        return threshold_grid_df, selected_thresholds_df, pd.DataFrame(), pd.DataFrame()

    selected_thresholds_df = selected_thresholds_df.merge(
        candidate_metadata_for_thresholds(panel_df),
        on="selected_config_id",
        how="left",
        validate="one_to_one",
    )

    metrics_df, objects_3way_df = evaluate_three_way_by_config(
        object_df=object_df,
        thresholds_df=selected_thresholds_df,
        config_id_col="selected_config_id",
        extra_group_cols=[
            "candidate_id",
            "model_candidate_id",
            "matrix_family",
            "matrix_method",
            "training_matrix_id",
            "preprocessing",
            "rule_variant",
            "n_components",
            "alpha",
            "object_threshold",
        ],
        target_class=TARGET_CLASS,
        non_target_label=NON_TARGET_LABEL,
    )

    if len(objects_3way_df) > 0:
        objects_3way_df = add_three_way_confidence(
            objects_3way_df,
            target_class=TARGET_CLASS,
            non_target_label=NON_TARGET_LABEL,
        )

    if len(metrics_df) > 0:
        metrics_df = metrics_df.merge(
            selected_thresholds_df[
                [
                    "selected_config_id",
                    "three_way_lower_threshold",
                    "three_way_upper_threshold",
                ]
            ],
            on="selected_config_id",
            how="left",
            validate="many_to_one",
        )
        metrics_df["fn_rate"] = metrics_df["target_miss_rate"]
        metrics_df["fp_rate"] = metrics_df["non_target_false_accept_rate"]
        metrics_df["balanced_accuracy"] = metrics_df["decided_balanced_accuracy"]
        metrics_df = finalize_metric_table(metrics_df, decision_mode="3way", metric_level="object")

    return threshold_grid_df, selected_thresholds_df, metrics_df, objects_3way_df


refit_panel_df = candidate_panel_df.copy()
if MAX_REFIT_CANDIDATES is not None:
    refit_panel_df = refit_panel_df.head(int(MAX_REFIT_CANDIDATES)).copy()
    print(f"Debug mode: refitting only {len(refit_panel_df)} candidates.")


In [ ]:
validation_refit_source = "computed_refit" if RUN_REFIT else "existing_parquet"

if RUN_REFIT:
    validation_refit_metric_parts = []
    validation_2way_object_metric_parts = []
    validation_2way_pixel_metric_parts = []
    validation_3way_threshold_grid_parts = []
    validation_3way_selected_threshold_parts = []
    validation_3way_object_metric_parts = []
    validation_refit_pixel_error_parts = []
    validation_refit_error_parts = []
    validation_refit_object_parts = []
    validation_refit_pixel_parts = []
    validation_refit_3way_object_parts = []
    batch_manifest_rows = []

    for batch_id, start, stop, batch_df in iter_candidate_batches(refit_panel_df, REFIT_BATCH_SIZE):
        print(f"[{EVALUATION_STAGE}] {batch_id}: candidates {start + 1}-{stop} / {len(refit_panel_df)}")
        batch_paths = {}

        try:
            (
                batch_refit_metrics_df,
                batch_objects_df,
                batch_pixels_df,
                batch_pixel_errors_df,
                batch_errors_df,
            ) = refit_selected_simca_configs(
                selected_configs_df=batch_df,
                object_db=object_db,
                image_db=image_db,
                train_filters=TRAIN_FILTERS,
                projection_filters=VALIDATION_FILTERS,
                preprocessing_configs=preprocessing_configs_by_family,
                evaluation_split=EVALUATION_STAGE,
                wavelengths=wavelengths,
                random_state=RANDOM_STATE,
                replace=REPLACE_BALANCED_PIXELS,
                cv_n_splits=CV_N_SPLITS,
                cv_group_col=CV_GROUP_COL,
                target_class=TARGET_CLASS,
                non_target_label=NON_TARGET_LABEL,
            )
        except Exception as exc:
            batch_refit_metrics_df = pd.DataFrame()
            batch_objects_df = pd.DataFrame()
            batch_pixels_df = pd.DataFrame()
            batch_pixel_errors_df = pd.DataFrame()
            batch_errors_df = batch_df.copy()
            batch_errors_df["evaluation_split"] = EVALUATION_STAGE
            batch_errors_df["error"] = repr(exc)
            print("  -> BATCH ERROR:", repr(exc))

        batch_2way_object_metrics_df = build_2way_object_metrics(batch_refit_metrics_df) if len(batch_refit_metrics_df) else pd.DataFrame()
        batch_2way_pixel_metrics_df = build_2way_pixel_metrics(batch_pixels_df) if len(batch_pixels_df) else pd.DataFrame()
        (
            batch_3way_threshold_grid_df,
            batch_3way_selected_thresholds_df,
            batch_3way_object_metrics_df,
            batch_3way_objects_df,
        ) = build_3way_outputs(batch_objects_df, batch_df)

        if len(batch_refit_metrics_df):
            validation_refit_metric_parts.append(batch_refit_metrics_df)
        if len(batch_2way_object_metrics_df):
            validation_2way_object_metric_parts.append(batch_2way_object_metrics_df)
        if len(batch_2way_pixel_metrics_df):
            validation_2way_pixel_metric_parts.append(batch_2way_pixel_metrics_df)
        if len(batch_3way_threshold_grid_df):
            validation_3way_threshold_grid_parts.append(batch_3way_threshold_grid_df)
        if len(batch_3way_selected_thresholds_df):
            validation_3way_selected_threshold_parts.append(batch_3way_selected_thresholds_df)
        if len(batch_3way_object_metrics_df):
            validation_3way_object_metric_parts.append(batch_3way_object_metrics_df)
        if len(batch_pixel_errors_df):
            validation_refit_pixel_error_parts.append(batch_pixel_errors_df)
        if len(batch_errors_df):
            validation_refit_error_parts.append(batch_errors_df)

        if SAVE_COMBINED_OBJECT_TABLES and len(batch_objects_df):
            validation_refit_object_parts.append(batch_objects_df.copy())
        if SAVE_COMBINED_PIXEL_TABLES and len(batch_pixels_df):
            validation_refit_pixel_parts.append(batch_pixels_df.copy())
        if SAVE_COMBINED_3WAY_OBJECT_TABLES and len(batch_3way_objects_df):
            validation_refit_3way_object_parts.append(batch_3way_objects_df.copy())

        if SAVE_BATCH_METRIC_TABLES:
            batch_paths["refit_metrics_path"] = save_batch_table(
                batch_refit_metrics_df,
                VALIDATION_BATCH_METRICS_DIR,
                batch_id,
                "refit_metrics",
            )
            batch_paths["2way_object_metrics_path"] = save_batch_table(
                batch_2way_object_metrics_df,
                VALIDATION_BATCH_METRICS_DIR,
                batch_id,
                "2way_object_metrics",
            )
            batch_paths["2way_pixel_metrics_path"] = save_batch_table(
                batch_2way_pixel_metrics_df,
                VALIDATION_BATCH_METRICS_DIR,
                batch_id,
                "2way_pixel_metrics",
            )
            batch_paths["3way_threshold_grid_path"] = save_batch_table(
                batch_3way_threshold_grid_df,
                VALIDATION_BATCH_METRICS_DIR,
                batch_id,
                "3way_threshold_grid",
            )
            batch_paths["3way_selected_thresholds_path"] = save_batch_table(
                batch_3way_selected_thresholds_df,
                VALIDATION_BATCH_METRICS_DIR,
                batch_id,
                "3way_selected_thresholds",
            )
            batch_paths["3way_object_metrics_path"] = save_batch_table(
                batch_3way_object_metrics_df,
                VALIDATION_BATCH_METRICS_DIR,
                batch_id,
                "3way_object_metrics",
            )
        if SAVE_BATCH_OBJECT_TABLES:
            batch_paths["objects_path"] = save_batch_table(
                batch_objects_df,
                VALIDATION_BATCH_OBJECTS_DIR,
                batch_id,
                "objects",
            )
        if SAVE_BATCH_PIXEL_TABLES:
            batch_paths["pixels_path"] = save_batch_table(
                batch_pixels_df,
                VALIDATION_BATCH_PIXELS_DIR,
                batch_id,
                "pixels",
            )
        if SAVE_BATCH_3WAY_OBJECT_TABLES:
            batch_paths["objects_3way_path"] = save_batch_table(
                batch_3way_objects_df,
                VALIDATION_BATCH_3WAY_OBJECTS_DIR,
                batch_id,
                "objects_3way",
            )

        manifest_row = {
            "batch_id": batch_id,
            "row_start": int(start),
            "row_stop": int(stop),
            "n_candidates": int(len(batch_df)),
            "n_refit_metric_rows": int(len(batch_refit_metrics_df)),
            "n_2way_object_metric_rows": int(len(batch_2way_object_metrics_df)),
            "n_2way_pixel_metric_rows": int(len(batch_2way_pixel_metrics_df)),
            "n_3way_threshold_grid_rows": int(len(batch_3way_threshold_grid_df)),
            "n_3way_selected_threshold_rows": int(len(batch_3way_selected_thresholds_df)),
            "n_3way_object_metric_rows": int(len(batch_3way_object_metrics_df)),
            "n_object_rows": int(len(batch_objects_df)),
            "n_pixel_rows": int(len(batch_pixels_df)),
            "n_3way_object_rows": int(len(batch_3way_objects_df)),
            "n_pixel_error_rows": int(len(batch_pixel_errors_df)),
            "n_error_rows": int(len(batch_errors_df)),
        }
        manifest_row.update({
            key: str(value) if value is not None else None
            for key, value in batch_paths.items()
        })
        batch_manifest_rows.append(manifest_row)

        del batch_refit_metrics_df
        del batch_objects_df
        del batch_pixels_df
        del batch_pixel_errors_df
        del batch_errors_df
        del batch_2way_object_metrics_df
        del batch_2way_pixel_metrics_df
        del batch_3way_threshold_grid_df
        del batch_3way_selected_thresholds_df
        del batch_3way_object_metrics_df
        del batch_3way_objects_df
        gc.collect()

    validation_refit_metrics_df = concat_nonempty(validation_refit_metric_parts)
    validation_2way_object_metrics_df = concat_nonempty(validation_2way_object_metric_parts)
    validation_2way_pixel_metrics_df = concat_nonempty(validation_2way_pixel_metric_parts)
    validation_3way_threshold_grid_df = concat_nonempty(validation_3way_threshold_grid_parts)
    validation_3way_selected_thresholds_df = concat_nonempty(validation_3way_selected_threshold_parts)
    validation_3way_object_metrics_df = concat_nonempty(validation_3way_object_metric_parts)
    validation_refit_pixel_errors_by_image_df = concat_nonempty(validation_refit_pixel_error_parts)
    validation_refit_errors_df = concat_nonempty(validation_refit_error_parts)
    validation_refit_objects_df = concat_nonempty(validation_refit_object_parts)
    validation_refit_pixels_df = concat_nonempty(validation_refit_pixel_parts) if SAVE_COMBINED_PIXEL_TABLES else pd.DataFrame()
    validation_refit_3way_objects_df = concat_nonempty(validation_refit_3way_object_parts)
    validation_batch_manifest_df = pd.DataFrame(batch_manifest_rows)
else:
    if not USE_EXISTING_VALIDATION_REFIT_OUTPUTS:
        raise RuntimeError(
            "RUN_REFIT is False and USE_EXISTING_VALIDATION_REFIT_OUTPUTS is False. "
            "Enable RUN_REFIT or allow loading existing validation parquet outputs."
        )
    missing_existing = missing_existing_validation_refit_paths()
    if missing_existing:
        raise FileNotFoundError(
            "RUN_REFIT is False but required validation refit parquet outputs are missing:\n"
            + "\n".join(map(str, missing_existing))
        )
    loaded_outputs = load_existing_validation_refit_outputs()
    globals().update(loaded_outputs)
    print("Loaded existing validation refit outputs from:", RESULTS_DIR)


In [ ]:
# Refit outputs are computed or loaded in the previous cell.
print("Validation refit source:", validation_refit_source)


In [ ]:
required_validation_tables = {
    "validation_refit_metrics_df": validation_refit_metrics_df,
    "validation_2way_object_metrics_df": validation_2way_object_metrics_df,
    "validation_2way_pixel_metrics_df": validation_2way_pixel_metrics_df,
    "validation_3way_object_metrics_df": validation_3way_object_metrics_df,
}
empty_required = [
    name for name, df in required_validation_tables.items()
    if df is None or len(df) == 0
]
if empty_required:
    raise RuntimeError(
        "Validation refit outputs are incomplete. Empty required table(s): "
        + ", ".join(empty_required)
    )

if validation_refit_errors_df.empty:
    validation_refit_errors_df = pd.DataFrame(columns=["selected_config_id", "evaluation_split", "error"])
if validation_refit_pixel_errors_by_image_df.empty:
    validation_refit_pixel_errors_by_image_df = pd.DataFrame(columns=["selected_config_id", "evaluation_split", "source_image"])
if "validation_refit_objects_df" not in globals():
    validation_refit_objects_df = pd.DataFrame()
if "validation_refit_pixels_df" not in globals():
    validation_refit_pixels_df = pd.DataFrame()
if "validation_refit_3way_objects_df" not in globals():
    validation_refit_3way_objects_df = pd.DataFrame()
if "validation_batch_manifest_df" not in globals():
    validation_batch_manifest_df = pd.DataFrame()

print("Validation refit source:", validation_refit_source)
print("Validation refit metrics:", validation_refit_metrics_df.shape)
print("2-way object metrics:", validation_2way_object_metrics_df.shape)
print("2-way pixel metrics:", validation_2way_pixel_metrics_df.shape)
print("3-way object metrics:", validation_3way_object_metrics_df.shape)
print("Combined objects:", validation_refit_objects_df.shape)
print("Combined pixels:", validation_refit_pixels_df.shape)
print("Refit errors:", validation_refit_errors_df.shape)
display(validation_batch_manifest_df.head())
display(validation_2way_object_metrics_df.head())


## Optional duplicated-candidate refit check

Set `REFIT_DUPLICATED = True` to refit the independent `refit_panel_duplicated_df` table. This is a diagnostic run only: it checks whether candidates removed by the metric-equivalence pruning still produce identical validation metrics after refit.


In [ ]:
# Optional local override.
# Set REFIT_DUPLICATED in the configuration cell above. Do not override it here
# unless you intentionally want to change only the duplicated-refit block.
# REFIT_DUPLICATED = True


In [ ]:
DUPLICATED_REFIT_EVALUATION_STAGE = "validation_batch_3_duplicate_check_refit"


def build_duplicated_refit_metric_comparison(
    object_metrics_df,
    group_summary_df,
    group_col="metric_equivalence_group_id",
):
    if object_metrics_df is None or len(object_metrics_df) == 0:
        return pd.DataFrame()
    if group_col not in object_metrics_df.columns:
        return pd.DataFrame()

    metric_cols = [
        col for col in METRIC_EQUIVALENCE_METRIC_COLUMNS
        if col in object_metrics_df.columns
    ]
    rows = []
    grouped = object_metrics_df.dropna(subset=[group_col]).groupby(group_col, dropna=False)

    for group_id, group in grouped:
        row = {
            group_col: group_id,
            "n_refit_candidates": int(len(group)),
            "candidate_ids": ",".join(group["candidate_id"].astype(str))
            if "candidate_id" in group.columns else "",
            "selected_config_ids": ",".join(group["selected_config_id"].astype(str))
            if "selected_config_id" in group.columns else "",
        }

        all_equal_flags = []
        all_match_pre_flags = []
        for col in metric_cols:
            values = pd.to_numeric(group[col], errors="coerce")
            rounded = values.round(12)
            n_unique = int(rounded.nunique(dropna=False))
            row[f"{col}_post_refit_nunique"] = n_unique
            row[f"{col}_post_refit_min"] = float(values.min()) if len(values) else np.nan
            row[f"{col}_post_refit_max"] = float(values.max()) if len(values) else np.nan
            equal_after = n_unique <= 1
            row[f"{col}_post_refit_all_equal"] = bool(equal_after)
            all_equal_flags.append(equal_after)

            pre_col = f"pre_refit_{col}"
            if pre_col in group.columns:
                pre_values = pd.to_numeric(group[pre_col], errors="coerce")
                matches = np.isclose(
                    values.to_numpy(dtype=float),
                    pre_values.to_numpy(dtype=float),
                    rtol=1e-9,
                    atol=1e-12,
                    equal_nan=True,
                )
                match_all = bool(matches.all()) if len(matches) else False
                row[f"{col}_matches_pre_refit_all"] = match_all
                row[f"{col}_max_abs_delta_vs_pre_refit"] = float(
                    np.nanmax(np.abs(values.to_numpy(dtype=float) - pre_values.to_numpy(dtype=float)))
                ) if len(matches) else np.nan
                all_match_pre_flags.append(match_all)

        row["all_post_refit_metrics_equal"] = bool(all(all_equal_flags)) if all_equal_flags else False
        row["all_post_refit_metrics_match_pre_refit"] = (
            bool(all(all_match_pre_flags)) if all_match_pre_flags else False
        )
        rows.append(row)

    out = pd.DataFrame(rows)
    if group_summary_df is not None and len(group_summary_df) > 0 and group_col in group_summary_df.columns:
        meta_cols = [
            col for col in [
                group_col,
                "varied_parameter_group",
                "varied_columns",
                "n_metric_equivalent_candidates",
                "kept_candidate_id",
                "dropped_candidate_ids",
                "varied_values_json",
            ]
            if col in group_summary_df.columns
        ]
        out = out.merge(
            group_summary_df[meta_cols].drop_duplicates(group_col),
            on=group_col,
            how="left",
            validate="one_to_one",
        )
    return out


duplicated_refit_metrics_df = pd.DataFrame()
duplicated_refit_2way_object_metrics_df = pd.DataFrame(
    columns=list(expcfg.SIMCA_CANDIDATE_EVALUATION_REQUIRED_COLUMNS)
)
duplicated_refit_2way_pixel_metrics_df = pd.DataFrame(
    columns=list(expcfg.SIMCA_CANDIDATE_EVALUATION_REQUIRED_COLUMNS)
)
duplicated_refit_pixel_errors_by_image_df = pd.DataFrame()
duplicated_refit_metric_comparison_df = pd.DataFrame()
duplicated_refit_errors_df = pd.DataFrame(columns=["selected_config_id", "evaluation_split", "error"])
duplicated_refit_batch_manifest_df = pd.DataFrame()

duplicated_refit_panel_to_run_df = refit_panel_duplicated_df.copy()
if DUPLICATED_REFIT_MAX_GROUPS is not None and len(duplicated_refit_panel_to_run_df) > 0:
    selected_group_ids = (
        duplicated_refit_panel_to_run_df["metric_equivalence_group_id"]
        .dropna()
        .drop_duplicates()
        .head(int(DUPLICATED_REFIT_MAX_GROUPS))
    )
    duplicated_refit_panel_to_run_df = duplicated_refit_panel_to_run_df.loc[
        duplicated_refit_panel_to_run_df["metric_equivalence_group_id"].isin(selected_group_ids)
    ].reset_index(drop=True)

if REFIT_DUPLICATED and len(duplicated_refit_panel_to_run_df) > 0:
    duplicated_metric_parts = []
    duplicated_object_metric_parts = []
    duplicated_pixel_metric_parts = []
    duplicated_pixel_error_parts = []
    duplicated_error_parts = []
    duplicated_manifest_rows = []

    for batch_id, start, stop, batch_df in iter_candidate_batches(
        duplicated_refit_panel_to_run_df,
        DUPLICATED_REFIT_BATCH_SIZE,
    ):
        print(
            f"[{DUPLICATED_REFIT_EVALUATION_STAGE}] {batch_id}: "
            f"candidates {start + 1}-{stop} / {len(duplicated_refit_panel_to_run_df)}"
        )
        (
            batch_metrics_df,
            batch_objects_df,
            batch_pixels_df,
            batch_pixel_errors_df,
            batch_errors_df,
        ) = refit_selected_simca_configs(
            selected_configs_df=batch_df,
            object_db=object_db,
            image_db=image_db,
            train_filters=TRAIN_FILTERS,
            projection_filters=VALIDATION_FILTERS,
            preprocessing_configs=preprocessing_configs_by_family,
            evaluation_split=DUPLICATED_REFIT_EVALUATION_STAGE,
            wavelengths=wavelengths,
            random_state=RANDOM_STATE,
            replace=REPLACE_BALANCED_PIXELS,
            cv_n_splits=CV_N_SPLITS,
            cv_group_col=CV_GROUP_COL,
            target_class=TARGET_CLASS,
            non_target_label=NON_TARGET_LABEL,
        )

        batch_object_metrics_df = build_2way_object_metrics(batch_metrics_df) if len(batch_metrics_df) else pd.DataFrame()
        if len(batch_object_metrics_df):
            batch_object_metrics_df["evaluation_stage"] = DUPLICATED_REFIT_EVALUATION_STAGE
        batch_pixel_metrics_df = build_2way_pixel_metrics(batch_pixels_df) if len(batch_pixels_df) else pd.DataFrame()
        if len(batch_pixel_metrics_df):
            batch_pixel_metrics_df["evaluation_stage"] = DUPLICATED_REFIT_EVALUATION_STAGE

        duplicated_metric_parts.append(batch_metrics_df)
        duplicated_object_metric_parts.append(batch_object_metrics_df)
        duplicated_pixel_metric_parts.append(batch_pixel_metrics_df)
        duplicated_pixel_error_parts.append(batch_pixel_errors_df)
        duplicated_error_parts.append(batch_errors_df)
        duplicated_manifest_rows.append({
            "batch_id": batch_id,
            "row_start": int(start),
            "row_stop": int(stop),
            "n_candidates": int(len(batch_df)),
            "n_refit_metric_rows": int(len(batch_metrics_df)),
            "n_2way_object_metric_rows": int(len(batch_object_metrics_df)),
            "n_2way_pixel_metric_rows": int(len(batch_pixel_metrics_df)),
            "n_pixel_error_rows": int(len(batch_pixel_errors_df)),
            "n_error_rows": int(len(batch_errors_df)),
        })

        del batch_metrics_df
        del batch_objects_df
        del batch_pixels_df
        del batch_pixel_errors_df
        del batch_errors_df
        del batch_object_metrics_df
        del batch_pixel_metrics_df
        gc.collect()

    duplicated_refit_metrics_df = concat_nonempty(duplicated_metric_parts)
    duplicated_refit_2way_object_metrics_df = concat_nonempty(duplicated_object_metric_parts)
    duplicated_refit_2way_pixel_metrics_df = concat_nonempty(duplicated_pixel_metric_parts)
    duplicated_refit_pixel_errors_by_image_df = concat_nonempty(duplicated_pixel_error_parts)
    duplicated_refit_errors_df = concat_nonempty(duplicated_error_parts)
    duplicated_refit_batch_manifest_df = pd.DataFrame(duplicated_manifest_rows)
    duplicated_refit_metric_comparison_df = build_duplicated_refit_metric_comparison(
        duplicated_refit_2way_object_metrics_df,
        metric_equivalence_groups_df,
    )
else:
    if len(refit_panel_duplicated_df) == 0:
        print("No duplicated/equivalent candidate group available for optional refit.")
    else:
        print("Skipped duplicated refit check because REFIT_DUPLICATED is False.")

if duplicated_refit_errors_df.empty:
    duplicated_refit_errors_df = pd.DataFrame(columns=["selected_config_id", "evaluation_split", "error"])

duplicated_refit_saved_paths = []
if REFIT_DUPLICATED or len(refit_panel_duplicated_df) == 0:
    for df, path in [
        (duplicated_refit_2way_object_metrics_df, DUPLICATED_REFIT_2WAY_OBJECT_METRICS_PATH),
        (duplicated_refit_2way_pixel_metrics_df, DUPLICATED_REFIT_2WAY_PIXEL_METRICS_PATH),
        (duplicated_refit_pixel_errors_by_image_df, DUPLICATED_REFIT_PIXEL_ERRORS_BY_IMAGE_PATH),
        (duplicated_refit_metric_comparison_df, DUPLICATED_REFIT_METRIC_COMPARISON_PATH),
        (duplicated_refit_errors_df, DUPLICATED_REFIT_ERRORS_PATH),
        (duplicated_refit_batch_manifest_df, DUPLICATED_REFIT_BATCH_MANIFEST_PATH),
    ]:
        duplicated_refit_saved_paths.append(save_parquet(compact_simca_table_for_path(df, path), path))
else:
    print("REFIT_DUPLICATED is False; existing duplicated-refit parquet outputs were left unchanged.")

print("Duplicated refit panel:", refit_panel_duplicated_df.shape)
print("Duplicated refit panel to run:", duplicated_refit_panel_to_run_df.shape)
print("Duplicated object metrics:", duplicated_refit_2way_object_metrics_df.shape)
print("Duplicated comparison:", duplicated_refit_metric_comparison_df.shape)
if len(duplicated_refit_metric_comparison_df) > 0:
    display(
        duplicated_refit_metric_comparison_df[
            [
                "metric_equivalence_group_id",
                "varied_parameter_group",
                "n_refit_candidates",
                "all_post_refit_metrics_equal",
                "all_post_refit_metrics_match_pre_refit",
            ]
        ].head()
    )


## Long metrics table and save

The consolidated metrics table is the stable output for downstream notebooks. The batch manifest records where detailed projection parts were saved and which optional combined tables were materialized.


In [ ]:
validation_metrics_long_df = pd.concat(
    [
        validation_2way_object_metrics_df,
        validation_2way_pixel_metrics_df,
        validation_3way_object_metrics_df,
    ],
    ignore_index=True,
    sort=False,
)

expected_tracks = set(expcfg.SIMCA_SELECTION_TRACKS)
observed_tracks = set(validation_metrics_long_df["selection_track"].dropna().astype(str))
missing_tracks = sorted(expected_tracks - observed_tracks)
if MAX_REFIT_CANDIDATES is None and missing_tracks:
    raise RuntimeError(f"Missing expected SIMCA selection tracks in 04C metrics: {missing_tracks}")
if MAX_REFIT_CANDIDATES is not None and missing_tracks:
    print(f"Debug subset warning: missing tracks are allowed with MAX_REFIT_CANDIDATES={MAX_REFIT_CANDIDATES}: {missing_tracks}")

validation_diagnostics_df = (
    validation_metrics_long_df
    .groupby(
        ["selection_track", "matrix_family", "decision_mode", "metric_level"],
        dropna=False,
    )
    .agg(
        n_rows=("candidate_id", "size"),
        n_candidates=("candidate_id", "nunique"),
        best_fn_rate=("fn_rate", "min"),
        best_fp_rate=("fp_rate", "min"),
        best_balanced_accuracy=("balanced_accuracy", "max"),
    )
    .reset_index()
    .sort_values(["selection_track", "metric_level"])
    .reset_index(drop=True)
)

validation_protocol_df = pd.DataFrame([{
    "notebook": "04C_simca_concat_refit",
    "results_tag": RESULTS_TAG,
    "db_h5_path": str(DB_H5_PATH),
    "pca_selected_preprocessings_path": str(PCA_SELECTED_PREPROCESSINGS_PATH),
    "grid_model_candidates_path": str(GRID_MODEL_CANDIDATES_PATH),
    "optuna_new_model_candidates_path": str(OPTUNA_NEW_MODEL_CANDIDATES_PATH),
    "train_filters_json": json.dumps(TRAIN_FILTERS, default=str),
    "validation_filters_json": json.dumps(VALIDATION_FILTERS, default=str),
    "three_way_lower_thresholds_json": json.dumps([float(x) for x in THREE_WAY_LOWER_THRESHOLDS]),
    "three_way_upper_thresholds_json": json.dumps([float(x) for x in THREE_WAY_UPPER_THRESHOLDS]),
    "max_three_way_target_miss_rate": float(MAX_THREE_WAY_TARGET_MISS_RATE),
    "max_three_way_false_accept_rate": float(MAX_THREE_WAY_FALSE_ACCEPT_RATE),
    "max_three_way_uncertain_rate": float(MAX_THREE_WAY_UNCERTAIN_RATE),
    "run_refit": bool(RUN_REFIT),
"use_existing_validation_refit_outputs": bool(USE_EXISTING_VALIDATION_REFIT_OUTPUTS),
"validation_refit_source": validation_refit_source,
"refit_batch_size": int(REFIT_BATCH_SIZE),
    "max_refit_candidates": None if MAX_REFIT_CANDIDATES is None else int(MAX_REFIT_CANDIDATES),
    "deduplicate_refit_configs": bool(DEDUPLICATE_REFIT_CONFIGS),
    "refit_config_dedup_columns_json": json.dumps(REFIT_CONFIG_DEDUP_COLUMNS),
    "drop_metric_equivalent_configs": bool(DROP_METRIC_EQUIVALENT_CONFIGS),
    "refit_duplicated": bool(REFIT_DUPLICATED),
    "duplicated_refit_batch_size": int(DUPLICATED_REFIT_BATCH_SIZE),
    "duplicated_refit_max_groups": None if DUPLICATED_REFIT_MAX_GROUPS is None else int(DUPLICATED_REFIT_MAX_GROUPS),
    "metric_equivalence_metric_columns_json": json.dumps(METRIC_EQUIVALENCE_METRIC_COLUMNS),
    "metric_equivalence_protected_columns_json": json.dumps(METRIC_EQUIVALENCE_PROTECTED_COLUMNS),
    "metric_equivalence_parameter_groups_json": json.dumps(METRIC_EQUIVALENCE_PARAMETER_GROUPS),
    "metric_equivalence_preference_note": METRIC_EQUIVALENCE_PREFERENCE_NOTE,
    "n_candidate_panel_before_refit_dedup": int(len(candidate_panel_before_refit_dedup_df)),
    "n_refit_config_duplicates_dropped": int(len(refit_config_duplicates_df)),
    "n_refit_config_groups": int(len(refit_config_dedup_summary_df)),
    "n_candidate_panel_after_refit_dedup": int(len(candidate_panel_after_refit_dedup_df)),
    "n_metric_equivalent_duplicates_dropped": int(len(metric_equivalence_dropped_df)),
    "n_metric_equivalence_groups": int(len(metric_equivalence_groups_df)),
    "n_refit_panel_duplicated": int(len(refit_panel_duplicated_df)),
    "n_duplicated_refit_panel_to_run": int(len(duplicated_refit_panel_to_run_df)),
    "n_duplicated_refit_object_metrics": int(len(duplicated_refit_2way_object_metrics_df)),
    "n_duplicated_refit_comparison_rows": int(len(duplicated_refit_metric_comparison_df)),
    "save_batch_metric_tables": bool(SAVE_BATCH_METRIC_TABLES),
    "save_batch_object_tables": bool(SAVE_BATCH_OBJECT_TABLES),
    "save_batch_pixel_tables": bool(SAVE_BATCH_PIXEL_TABLES),
    "save_batch_3way_object_tables": bool(SAVE_BATCH_3WAY_OBJECT_TABLES),
    "save_combined_object_tables": bool(SAVE_COMBINED_OBJECT_TABLES),
    "save_combined_pixel_tables": bool(SAVE_COMBINED_PIXEL_TABLES),
    "save_combined_3way_object_tables": bool(SAVE_COMBINED_3WAY_OBJECT_TABLES),
    "n_candidate_panel": int(len(candidate_panel_df)),
    "n_refit_panel": int(len(refit_panel_df)),
    "n_validation_2way_object_metrics": int(len(validation_2way_object_metrics_df)),
    "n_validation_2way_pixel_metrics": int(len(validation_2way_pixel_metrics_df)),
    "n_validation_3way_object_metrics": int(len(validation_3way_object_metrics_df)),
    "n_validation_refit_errors": int(len(validation_refit_errors_df)),
    "candidate_panel_path": str(CANDIDATE_PANEL_PATH),
    "refit_config_dedup_summary_path": str(REFIT_CONFIG_DEDUP_SUMMARY_PATH),
    "refit_config_duplicates_path": str(REFIT_CONFIG_DUPLICATES_PATH),
    "metric_equivalence_groups_path": str(METRIC_EQUIVALENCE_GROUPS_PATH),
    "metric_equivalence_dropped_path": str(METRIC_EQUIVALENCE_DROPPED_PATH),
    "duplicated_refit_panel_path": str(DUPLICATED_REFIT_PANEL_PATH),
    "duplicated_refit_2way_object_metrics_path": str(DUPLICATED_REFIT_2WAY_OBJECT_METRICS_PATH),
    "duplicated_refit_2way_pixel_metrics_path": str(DUPLICATED_REFIT_2WAY_PIXEL_METRICS_PATH),
    "duplicated_refit_metric_comparison_path": str(DUPLICATED_REFIT_METRIC_COMPARISON_PATH),
    "duplicated_refit_errors_path": str(DUPLICATED_REFIT_ERRORS_PATH),
    "duplicated_refit_batch_manifest_path": str(DUPLICATED_REFIT_BATCH_MANIFEST_PATH),
    "validation_metrics_long_path": str(VALIDATION_METRICS_LONG_PATH),
    "validation_batch_manifest_path": str(VALIDATION_BATCH_MANIFEST_PATH),
    "validation_batch_dir": str(VALIDATION_BATCH_DIR),
}])

saved_paths = []
for df, path in [
    (validation_2way_object_metrics_df, VALIDATION_2WAY_OBJECT_METRICS_PATH),
    (validation_2way_pixel_metrics_df, VALIDATION_2WAY_PIXEL_METRICS_PATH),
    (validation_3way_threshold_grid_df, VALIDATION_3WAY_THRESHOLD_GRID_PATH),
    (validation_3way_selected_thresholds_df, VALIDATION_3WAY_SELECTED_THRESHOLDS_PATH),
    (validation_3way_object_metrics_df, VALIDATION_3WAY_OBJECT_METRICS_PATH),
    (validation_metrics_long_df, VALIDATION_METRICS_LONG_PATH),
    (validation_refit_pixel_errors_by_image_df, VALIDATION_PIXEL_ERRORS_BY_IMAGE_PATH),
    (validation_refit_errors_df, VALIDATION_REFIT_ERRORS_PATH),
    (refit_config_dedup_summary_df, REFIT_CONFIG_DEDUP_SUMMARY_PATH),
    (refit_config_duplicates_df, REFIT_CONFIG_DUPLICATES_PATH),
    (metric_equivalence_groups_df, METRIC_EQUIVALENCE_GROUPS_PATH),
    (metric_equivalence_dropped_df, METRIC_EQUIVALENCE_DROPPED_PATH),
    (validation_diagnostics_df, VALIDATION_DIAGNOSTICS_PATH),
    (validation_protocol_df, VALIDATION_PROTOCOL_PATH),
    (validation_batch_manifest_df, VALIDATION_BATCH_MANIFEST_PATH),
]:
    saved_paths.append(save_parquet(compact_simca_table_for_path(df, path), path))

if SAVE_COMBINED_OBJECT_TABLES and len(validation_refit_objects_df) > 0:
    saved_paths.append(save_parquet(compact_simca_table_for_path(validation_refit_objects_df, VALIDATION_OBJECTS_PATH), VALIDATION_OBJECTS_PATH))
if SAVE_COMBINED_PIXEL_TABLES and len(validation_refit_pixels_df) > 0:
    saved_paths.append(save_parquet(compact_simca_table_for_path(validation_refit_pixels_df, VALIDATION_PIXELS_PATH), VALIDATION_PIXELS_PATH))
if SAVE_COMBINED_3WAY_OBJECT_TABLES and len(validation_refit_3way_objects_df) > 0:
    saved_paths.append(save_parquet(compact_simca_table_for_path(validation_refit_3way_objects_df, VALIDATION_3WAY_OBJECTS_PATH), VALIDATION_3WAY_OBJECTS_PATH))

print("Saved:")
for path in saved_paths:
    print(" -", path)

if not SAVE_COMBINED_PIXEL_TABLES:
    print("Skipped global pixel projection table:", VALIDATION_PIXELS_PATH)
    print("Enable SAVE_BATCH_PIXEL_TABLES=True only for a small candidate subset if detailed pixel projections are needed.")

display(validation_diagnostics_df)
display(list_result_files(RESULTS_DIR).head(30))


In [ ]:
# Outputs are saved in the previous cell.
